# nnUNet 3D Inference
Semantic segmentation → 3D connected components → instance masks

> **Kernel**: use the **`bw_env`** kernel (nnunetv2 lives there, not in `thesis_microglomeruli`).

In [ ]:
import os

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION — change these two lines only
# ─────────────────────────────────────────────────────────────────────────────
MODEL_FOLDER = "/work/scratch/rawat/models/nnunet/Dataset002_Calyx_Original/nnUNetTrainer__nnUNetPlans__3d_fullres"
INPUT_FOLDER = "/home/students/rawat/Final_dataset/Original/test/images"
# ─────────────────────────────────────────────────────────────────────────────

# Other model folders:
#   /work/scratch/rawat/models/nnunet/Dataset003_Calyx_DOG/nnUNetTrainer__nnUNetPlans__3d_fullres
#   /work/scratch/rawat/models/nnunet/Dataset004_Calyx_PSF/nnUNetTrainer__nnUNetPlans__3d_fullres
#   /work/scratch/rawat/models/nnunet/Dataset001_Calyx_All/nnUNetTrainer__nnUNetPlans__3d_fullres

OUTPUT_BASE            = "/home/students/rawat/Results"
FOLD                   = 0
CHECKPOINT             = "checkpoint_best.pth"
VALID_XY_SIZES         = {1100, 1101}
BLOB_REMOVAL_THRESHOLD = 1450

# Voxel spacing (must match training NIfTI headers)
Z_VOXEL_UM  = 0.3
XY_VOXEL_UM = 0.0709

# Set env vars BEFORE importing nnunetv2
os.environ["nnUNet_raw"]          = "/work/scratch/rawat/nnunet/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/work/scratch/rawat/nnunet/nnUNet_preprocessed"
os.environ["nnUNet_results"]      = "/work/scratch/rawat/models/nnunet"

MODEL_NAME = os.path.basename(os.path.dirname(MODEL_FOLDER))  # e.g. Dataset002_Calyx_Original

print(f"Model folder : {MODEL_FOLDER}")
print(f"Model name   : {MODEL_NAME}")
print(f"Input folder : {INPUT_FOLDER}")
print(f"Output base  : {OUTPUT_BASE}/{MODEL_NAME}/")

In [ ]:
import tempfile
import shutil
from pathlib import Path

import numpy as np
import nibabel as nib
import tifffile
import torch
from scipy.ndimage import label as ndimage_label, generate_binary_structure
from skimage.morphology import remove_small_objects

from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def to_zcyx(arr: np.ndarray) -> np.ndarray:
    """Normalise any TIFF to (Z, C, Y, X)."""
    if arr.ndim == 3:
        return arr[:, np.newaxis]          # single-channel
    if arr.shape[0] <= 4 and arr.shape[0] < arr.shape[1]:
        arr = arr.transpose(1, 0, 2, 3)   # (C,Z,Y,X) → (Z,C,Y,X)
        print(f"  [fix] Transposed (C,Z,Y,X) → (Z,C,Y,X): {arr.shape}")
    return arr


def tiff_to_nifti(tiff_path: Path, out_dir: Path, z_um: float, xy_um: float) -> list:
    """Convert a multi-channel TIFF to per-channel NIfTI files for nnUNet.

    Returns a list of NIfTI file paths [ch0, ch1, ...] in nnUNet naming convention
    (stem_0000.nii.gz, stem_0001.nii.gz, ...).
    """
    arr = tifffile.imread(str(tiff_path))   # whatever shape
    arr = to_zcyx(arr)                      # → (Z, C, Y, X)
    n_channels = arr.shape[1]

    # Affine encodes voxel size; nnUNet reads header.get_zooms()[:3]
    affine = np.diag([z_um, xy_um, xy_um, 1.0])

    stem = tiff_path.stem
    paths = []
    for c in range(n_channels):
        channel_data = arr[:, c].astype(np.float32)   # (Z, Y, X)
        nii = nib.Nifti1Image(channel_data, affine)
        out_path = out_dir / f"{stem}_{c:04d}.nii.gz"
        nib.save(nii, str(out_path))
        paths.append(str(out_path))

    return paths


print("Helper functions ready.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SCAN FOLDER FOR VALID IMAGES
# ─────────────────────────────────────────────────────────────────────────────
all_tifs = sorted(Path(INPUT_FOLDER).glob("*.tif"))
valid_images = []

print(f"Scanning: {INPUT_FOLDER}\n")
for p in all_tifs:
    arr = tifffile.imread(str(p))
    y, x = arr.shape[-2], arr.shape[-1]
    if y in VALID_XY_SIZES and x in VALID_XY_SIZES:
        valid_images.append(p)
        print(f"  VALID   {p.name}  {arr.shape}")
    else:
        print(f"  SKIP    {p.name}  {arr.shape}  (XY={y}x{x})")

print(f"\n{len(valid_images)} image(s) will be processed.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LOAD MODEL ONCE
# ─────────────────────────────────────────────────────────────────────────────
print(f"Loading predictor from: {MODEL_FOLDER}")

predictor = nnUNetPredictor(
    tile_step_size=0.5,
    use_gaussian=True,
    use_mirroring=True,
    perform_everything_on_device=True,
    device=DEVICE,
    verbose=False,
)
predictor.initialize_from_trained_model_folder(
    MODEL_FOLDER,
    use_folds=(FOLD,),
    checkpoint_name=CHECKPOINT,
)
print("Predictor ready.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# INFERENCE LOOP — all valid images
# ─────────────────────────────────────────────────────────────────────────────
structure = generate_binary_structure(3, 3)   # 26-connectivity

for img_path in valid_images:
    print(f"\n{'─'*60}")
    print(f"Processing: {img_path.name}")

    tmp_in  = Path(tempfile.mkdtemp(prefix="nnunet_in_"))
    tmp_out = Path(tempfile.mkdtemp(prefix="nnunet_out_"))
    try:
        # ── Convert TIFF → per-channel NIfTI ─────────────────────────────────
        channel_files = tiff_to_nifti(img_path, tmp_in, Z_VOXEL_UM, XY_VOXEL_UM)
        print(f"  NIfTI inputs : {[Path(f).name for f in channel_files]}")

        # ── nnUNet predict ────────────────────────────────────────────────────
        # predict_from_files takes list-of-lists for multi-channel cases
        predictor.predict_from_files(
            [channel_files],
            str(tmp_out),
            save_probabilities=False,
            overwrite=True,
            num_processes_preprocessing=1,
            num_processes_segmentation_export=1,
        )

        # ── Load predicted binary mask ────────────────────────────────────────
        pred_nii_path = next(tmp_out.glob("*.nii.gz"))
        binary = nib.load(str(pred_nii_path)).get_fdata().astype(bool)  # (Z,Y,X)

        # ── 3D connected components → instance segmentation ───────────────────
        labeled, n_obj = ndimage_label(binary, structure=structure)
        print(f"  After 3D CC        : {n_obj} objects")

        # ── Blob removal ──────────────────────────────────────────────────────
        clean = remove_small_objects(labeled.astype(bool), min_size=BLOB_REMOVAL_THRESHOLD, connectivity=1)
        clean, n_final = ndimage_label(clean, structure=structure)
        print(f"  After blob removal : {n_final} objects  (threshold={BLOB_REMOVAL_THRESHOLD} vox)")

        # ── Save ──────────────────────────────────────────────────────────────
        out_path = Path(OUTPUT_BASE) / MODEL_NAME / f"{img_path.stem}_mask.tif"
        os.makedirs(out_path.parent, exist_ok=True)
        dtype = np.uint32 if n_final > 65535 else np.uint16
        tifffile.imwrite(str(out_path), clean.astype(dtype))
        print(f"  Saved → {out_path}")

    finally:
        shutil.rmtree(tmp_in,  ignore_errors=True)
        shutil.rmtree(tmp_out, ignore_errors=True)

print(f"\n{'='*60}")
print(f"Done. Processed {len(valid_images)} image(s).")